In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import matplotlib.pyplot as plt
import numpy as np
import torch
import pyro
import pyro.distributions as dist
from pyro.infer import MCMC, NUTS
import pandas as pd
import warnings

warnings.filterwarnings('ignore')
pyro.set_rng_seed(179)
pyro.clear_param_store()
pd.set_option('display.max_columns', None)

# Load data

In [ ]:
games = pd.read_csv("matchups.csv")
team_logs = pd.read_csv("team_logs.csv")

required_cols = ["HOME_TEAM_ID", "AWAY_TEAM_ID", "HOME_WIN", "SEASON"]
missing_cols = [col for col in required_cols if col not in games.columns]
if missing_cols:
    raise ValueError(f"matchups.csv is missing required columns: {missing_cols}")

games = games.dropna(subset=required_cols).reset_index(drop=True)
games["HOME_WIN"] = games["HOME_WIN"].astype(int)
games["HOME_TEAM_ID"] = games["HOME_TEAM_ID"].astype(int)
games["AWAY_TEAM_ID"] = games["AWAY_TEAM_ID"].astype(int)

home_names = games[["HOME_TEAM_ID", "HOME_TEAM_NAME"]].rename(columns={"HOME_TEAM_ID": "TEAM_ID", "HOME_TEAM_NAME": "TEAM_NAME"})
away_names = games[["AWAY_TEAM_ID", "AWAY_TEAM_NAME"]].rename(columns={"AWAY_TEAM_ID": "TEAM_ID", "AWAY_TEAM_NAME": "TEAM_NAME"})
team_names = pd.concat([home_names, away_names, team_logs[["TEAM_ID", "TEAM_NAME"]]], ignore_index=True).dropna()
team_name_map = team_names.drop_duplicates("TEAM_ID").set_index("TEAM_ID")["TEAM_NAME"].to_dict()
season_label = ", ".join(map(str, sorted(games["SEASON"].unique())))

print(f"Games loaded: {len(games)}")
print(f"Seasons covered: {season_label}")
print(f"Home win rate: {games['HOME_WIN'].mean():.3f}")
display(games.head())

# Prepare Model Inputs

In [ ]:
# map NBA team IDs to 0..N-1 indices for the strength vector
all_team_ids = sorted(set(games["HOME_TEAM_ID"]) | set(games["AWAY_TEAM_ID"]))

team_id_to_idx = {tid: i for i, tid in enumerate(all_team_ids)}
idx_to_team_id = {i: tid for tid, i in team_id_to_idx.items()}
idx_to_team    = {i: team_name_map.get(tid, str(tid)) for i, tid in idx_to_team_id.items()}
team_to_idx    = {name: i for i, name in idx_to_team.items()}

home_idx = torch.tensor([team_id_to_idx[t] for t in games["HOME_TEAM_ID"]], dtype=torch.long)
away_idx = torch.tensor([team_id_to_idx[t] for t in games["AWAY_TEAM_ID"]], dtype=torch.long)
outcomes = torch.tensor(games["HOME_WIN"].values, dtype=torch.float) 

num_teams = len(all_team_ids)
num_games  = len(games)
home_win_rate = outcomes.mean().item()

print(f"Number of teams: {num_teams}")
print(f"Number of games: {num_games}")
print(f"Home win rate:   {home_win_rate:.3f}")

# Define the Probabilistic Model

In [ ]:
#Bayesian latent strength model
#strengths[i] ~ Normal(0,1) per team, home_adv ~ Normal(0,1) global
#outcome ~ Bernoulli(sigmoid(strength_home - strength_away + home_adv))

def model(home_idx, away_idx, outcomes=None):
    num_teams = int(torch.max(torch.cat([home_idx, away_idx])).item()) + 1

    strengths = pyro.sample(
        'strengths',
        dist.Normal(torch.zeros(num_teams), torch.ones(num_teams)).to_event(1)
    )

    home_adv = pyro.sample('home_adv', dist.Normal(torch.tensor(0.), torch.tensor(1.)))

    logits = strengths[home_idx] - strengths[away_idx] + home_adv

    with pyro.plate('games', len(home_idx)):
        pyro.sample('obs', dist.Bernoulli(logits=logits), obs=outcomes)

# Run NUTS Inference

In [ ]:
nuts_kernel = NUTS(model)
mcmc = MCMC(nuts_kernel, num_samples=500, warmup_steps=200, num_chains=1)
mcmc.run(home_idx, away_idx, outcomes)
samples = mcmc.get_samples()

# Posterior Team Strength Rankings

In [ ]:
#posterior mean and std per team from the 500 NUTS samples
strength_samples = samples['strengths'].detach().cpu()
posterior_mean = strength_samples.mean(dim=0).numpy()
posterior_std  = strength_samples.std(dim=0).numpy()

rankings = pd.DataFrame({
    'team_name':     [idx_to_team[idx] for idx in range(num_teams)],
    'mean_strength': posterior_mean,
    'std_strength':  posterior_std,
}).sort_values('mean_strength', ascending=False).reset_index(drop=True)

print('Top 10 teams by posterior mean strength:')
display(rankings.head(10))

print('Bottom 5 teams by posterior mean strength:')
display(rankings.tail(5))

# Visualize Team Strength Rankings

In [ ]:
# Horizontal bar chart of all teams ranked by posterior mean strength. Error bars = +/- 1 posterior std
plot_df = rankings.sort_values('mean_strength', ascending=True)

plt.figure(figsize=(10, 14))
plt.barh(
    plot_df['team_name'],
    plot_df['mean_strength'],
    xerr=plot_df['std_strength'],
    color='steelblue',
    alpha=0.85,
    ecolor='black',
    capsize=3,
)
plt.axvline(0, color='gray', linestyle='--', linewidth=1)
plt.title(f'Posterior Team Strength Estimates (NBA {season_label})')
plt.xlabel('Posterior mean latent strength')
plt.ylabel('Team')
plt.tight_layout()
plt.savefig('team_strengths.png', dpi=200, bbox_inches='tight')
plt.show()

# Home Court Advantage Posterior

In [ ]:
# posterior over home_adv with mean and 95% credible interval
home_adv_samples = samples['home_adv'].detach().cpu().numpy()
home_adv_mean = home_adv_samples.mean()
home_adv_ci   = np.percentile(home_adv_samples, [2.5, 97.5])

plt.figure(figsize=(8, 5))
plt.hist(home_adv_samples, bins=30, color='seagreen', alpha=0.8, edgecolor='white')
plt.axvline(home_adv_mean,   color='black', linestyle='-',  linewidth=2, label='Posterior mean')
plt.axvline(home_adv_ci[0],  color='black', linestyle='--', linewidth=1, label='95% credible interval')
plt.axvline(home_adv_ci[1],  color='black', linestyle='--', linewidth=1)
plt.title('Posterior Distribution of Home-Court Advantage')
plt.xlabel('Home-court advantage log-odds')
plt.ylabel('Posterior sample count')
plt.legend()
plt.tight_layout()
plt.show()

print(f'Posterior mean home-court advantage: {home_adv_mean:.3f}')
print(f'95% credible interval: [{home_adv_ci[0]:.3f}, {home_adv_ci[1]:.3f}]')

# Uncertainty vs. Data Size Analysis

In [ ]:
# compare posterior uncertainty to games observed, more data should mean narrower posteriors

home_counts  = games.groupby("HOME_TEAM_ID").size()
away_counts  = games.groupby("AWAY_TEAM_ID").size()
games_by_team = home_counts.add(away_counts, fill_value=0)

label_by_team_id = {tid: name.split()[-1] for tid, name in team_name_map.items()}

uncertainty_df = pd.DataFrame({
    'team_id':      [idx_to_team_id[idx] for idx in range(num_teams)],
    'team_name':    [idx_to_team[idx] for idx in range(num_teams)],
    'n_games':      [games_by_team.loc[idx_to_team_id[idx]] for idx in range(num_teams)],
    'posterior_std': posterior_std,
})
uncertainty_df['label'] = uncertainty_df['team_id'].map(label_by_team_id)

x = uncertainty_df['n_games'].to_numpy(dtype=float)
y = uncertainty_df['posterior_std'].to_numpy(dtype=float)

# trend line
slope, intercept = np.polyfit(x, y, deg=1)
x_line = np.linspace(x.min(), x.max(), 100)
y_line = slope * x_line + intercept

plt.figure(figsize=(10, 7))
plt.scatter(x, y, color='darkorange', alpha=0.85)
plt.plot(x_line, y_line, color='black', linestyle='--', linewidth=2, label='Linear trend')

for _, row in uncertainty_df.iterrows():
    plt.annotate(row['label'], (row['n_games'], row['posterior_std']),
                 fontsize=8, xytext=(4, 3), textcoords='offset points')

plt.title('Posterior Uncertainty vs. Number of Games Observed')
plt.xlabel('Number of games observed')
plt.ylabel('Posterior standard deviation')
plt.legend()
plt.tight_layout()
plt.show()

#  Posterior Predictive Probability

In [ ]:
def predict_proba(home_team_name, away_team_name, samples, team_to_idx, idx_to_name=None):
    """
    Returns the posterior predictive probability that the home team wins.

    home_team_name : str  full team name
    away_team_name : str  full team name
    samples        : dict from mcmc.get_samples()
    team_to_idx    : dict mapping team name to integer index

    Returns: float in [0, 1], probability the home team wins
    """
    if home_team_name not in team_to_idx:
        raise KeyError(f'Unknown home team name: {home_team_name}')
    if away_team_name not in team_to_idx:
        raise KeyError(f'Unknown away team name: {away_team_name}')

    h = team_to_idx[home_team_name]
    a = team_to_idx[away_team_name]

    strength_samples  = samples['strengths']
    home_adv_samples  = samples['home_adv']

    if not torch.is_tensor(strength_samples):
        strength_samples = torch.as_tensor(strength_samples)
    if not torch.is_tensor(home_adv_samples):
        home_adv_samples = torch.as_tensor(home_adv_samples)

    logits = strength_samples[:, h] - strength_samples[:, a] + home_adv_samples
    probs  = torch.sigmoid(logits)
    return probs.mean().item()